In [2]:
import openai
import pandas as pd
from dotenv import load_dotenv
import os

# load the .env file and get the HLT_API_KEY (the API key used to access the model)
load_dotenv()
class_sci_api_key = os.getenv('HLT_API_KEY')

# create the client using openai- need api_key and base_url
client = openai.OpenAI( 
    api_key=class_sci_api_key, 
    base_url="https://ol.sci.pitt.edu" # LiteLLM Proxy is OpenAI compatible, Read More: https://docs.litellm.ai/docs/proxy/user_keys 
) 

# get the subtitles for the movies in the test set:
df = pd.read_csv('testSubs.csv')

# function to get the prompted summary for a movie:
def get_prompt_summary(movie_subtitles):
    # ensure movie_subtitles is in string format:
    movie_subtitles = str(movie_subtitles)

    # the prompt we'll be giving to the llama model:
    prompt = f"Given these subtitles for a movie, please generate a summary of the plot: {movie_subtitles}"

    # pass the required arguments and make the call to the model to generate the response:
    """ Make an API call to Pitt SCI LLM """
    response = client.chat.completions.create( 
        model = 'llama3.1',
        messages = [{ 
                "role": "user", 
                "content": prompt
        }] 
    ) 

    # access the actual response generated and return it:
    response_text = response.choices[0].message.content
    return response_text

In [4]:
# prompt the model for a summary for each movie entry:
df['prompted_summary'] = df['subtitles'].apply(get_prompt_summary)

# write new data to a new file:
df.to_csv('llamaPromptedSummaries.csv', index=False)